# CLIP image embeddings

The pinned vision encoder preprocesses RGB images, then projects them into 512 dimensions. For unit vectors, the dot product equals cosine similarity.


In [ ]:
import os, sys
from pathlib import Path
root = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'ARCHITECTURE.md').is_file())
sys.path.insert(0, str(root))
from pipeline.prepare_dataset import read_json
fixture = root / 'data/processed/fixture'
manifest_path = Path(os.environ.get('STYLIST_MANIFEST', fixture / 'manifest.json'))
manifest = read_json(manifest_path)
image_root = Path(os.environ.get('STYLIST_IMAGE_ROOT', fixture))
print('Dataset:', manifest['dataset'], '| Revision:', manifest['revision'])

In [ ]:
import numpy as np
from PIL import Image
from backend.app.ml.encoder import ClipEncoder, MODEL_ID, MODEL_REVISION
print(MODEL_ID, MODEL_REVISION)
os.environ.setdefault('HF_HOME', str(root / 'data/raw/hf-cache'))
encoder = ClipEncoder()
with Image.open(image_root / manifest['items'][0]['image']) as first, Image.open(image_root / manifest['items'][1]['image']) as second:
    vectors = encoder([first.convert('RGB'), second.convert('RGB')])
print('Dimensions:', vectors.shape, 'L2 norms:', np.linalg.norm(vectors, axis=1))
print('Pairwise cosine:', float(vectors[0] @ vectors[1]))